# 90 - Testes
1. Testes unitários do cliente HTTP
2. Chamada real às APIs

In [0]:
%pip install -q pytest==8.3.3

In [0]:
import sys
import pytest

sys.dont_write_bytecode = True
codigo_saida = pytest.main(["-v", "-p", "no:cacheprovider", "../tests"])
assert codigo_saida == 0, "Algum teste falhou: veja o relatório acima"

In [0]:
import logging
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_settings
from src.http_client import HttpClient, RateLimiter, RateLimitExceededError

os.environ["FOOTBALL_DATA_API_KEY"] = dbutils.secrets.get(scope="fc", key="football_data_api_key")
settings = load_settings()
print(settings)

football = HttpClient(
    timeout=settings.timeout_seconds,
    max_retries=settings.max_retries,
    max_retry_wait=settings.max_retry_wait_seconds,
    rate_limiter=RateLimiter(settings.football_calls_per_minute),
    headers={"X-Auth-Token": settings.football_api_key},
)
competicao = football.get_json(f"{settings.football_base_url}/competitions/{settings.competition}")
temporada = competicao["currentSeason"]
print(competicao["name"], "| temporada:", temporada["startDate"], "→", temporada["endDate"])

In [0]:
clima = HttpClient(
    timeout=settings.timeout_seconds,
    max_retries=settings.max_retries,
    max_retry_wait=settings.max_retry_wait_seconds,
)
try:
    dados = clima.get_json(
        settings.weather_base_url,
        params={
            "latitude": -23.55, "longitude": -46.63,
            "start_date": "2026-01-01", "end_date": "2026-01-03",
            "daily": "temperature_2m_max,precipitation_sum",
            "timezone": "America/Sao_Paulo",
        },
    )
    print("Open-Meteo OK:", dados["daily"])
except RateLimitExceededError as erro:
    print("Open-Meteo recusou, mas o pipeline NÃO travou:", erro)